# Exp 5: Robustness Evaluation — The Missing Benchmarks

ViT-5 claims to be a "drop-in upgrade for mid-2020s vision backbones" but reports
**zero** robustness benchmarks. Every comparable backbone paper (DeiT-III, ConvNeXt V2,
EVA-02, Vision Mamba) reports on ImageNet-C/A/R/Sketch.

We evaluate:
- **ImageNet-C** (corruption robustness — 15 corruptions x 5 severities)
- **ImageNet-A** (natural adversarial examples)
- **ImageNet-R** (renditions: art, cartoons, sketches)
- **ImageNet-Sketch** (sketch domain)

Since full ImageNet-scale OOD datasets are large, we use publicly available
HuggingFace datasets and evaluate on subsets where needed.

Models: ViT-5-Small vs DeiT-III-Small. Forward-pass only, T4-safe.

In [ ]:
!pip install -q timm einops huggingface_hub matplotlib datasets

In [ ]:
!git clone https://github.com/wangf3014/ViT-5.git vit5_repo 2>/dev/null || echo 'Already cloned'

In [ ]:
from huggingface_hub import hf_hub_download
import os

os.makedirs('checkpoints', exist_ok=True)
vit5_small_ckpt = hf_hub_download(
    repo_id='FengWang3211/ViT-5',
    filename='vit5_small_patch16_224.pth',
    local_dir='checkpoints'
)
print(f'Downloaded: {vit5_small_ckpt}')

In [ ]:
import sys
sys.path.insert(0, 'vit5_repo')

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torchvision import transforms
from PIL import Image
from collections import defaultdict
import json

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
from models_vit5 import vit5_small
import timm

def load_vit5_small():
    model = vit5_small(img_size=224)
    ckpt = torch.load(vit5_small_ckpt, map_location='cpu', weights_only=False)
    state_dict = ckpt['model'] if 'model' in ckpt else ckpt
    model.load_state_dict(state_dict, strict=False)
    return model.to(device).eval()

def load_deit3_small():
    model = timm.create_model('deit3_small_patch16_224.fb_in1k', pretrained=True)
    return model.to(device).eval()

print('Model loaders ready.')

In [ ]:
# ============================================================
# Standard ImageNet preprocessing
# ============================================================

transform_eval = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print('Transforms ready.')

In [ ]:
# ============================================================
# Generic evaluation function
# ============================================================

@torch.no_grad()
def evaluate_accuracy(model, dataloader, topk=(1, 5)):
    """Compute top-k accuracy."""
    correct = {k: 0 for k in topk}
    total = 0
    
    for batch_idx, (images, labels) in enumerate(dataloader):
        images = images.to(device)
        labels = labels.to(device)
        
        logits = model(images)
        
        for k in topk:
            _, topk_preds = logits.topk(k, dim=1)
            correct[k] += (topk_preds == labels.unsqueeze(1)).any(dim=1).sum().item()
        total += labels.size(0)
        
        if (batch_idx + 1) % 20 == 0:
            print(f'    Batch {batch_idx+1}/{len(dataloader)} '
                  f'(top1={100*correct[1]/total:.2f}%)', end='\r')
    
    print()  # newline after \r
    return {k: 100.0 * correct[k] / total for k in topk}


def evaluate_model_on_dataset(load_fn, model_name, dataloader):
    """Load model, evaluate, free GPU."""
    print(f'  Loading {model_name}...')
    model = load_fn()
    
    print(f'  Evaluating {model_name}...')
    results = evaluate_accuracy(model, dataloader)
    
    del model
    torch.cuda.empty_cache()
    
    print(f'  {model_name}: Top-1={results[1]:.2f}%, Top-5={results[5]:.2f}%')
    return results

print('Evaluation functions ready.')

In [ ]:
# ============================================================
# Download and prepare OOD datasets from HuggingFace
# ============================================================

from datasets import load_dataset

class HFImageDataset(torch.utils.data.Dataset):
    """Wraps a HuggingFace dataset for PyTorch DataLoader."""
    def __init__(self, hf_dataset, transform, image_key='image', label_key='label', max_samples=None):
        self.dataset = hf_dataset
        self.transform = transform
        self.image_key = image_key
        self.label_key = label_key
        if max_samples is not None:
            self.dataset = self.dataset.select(range(min(max_samples, len(self.dataset))))
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = item[self.image_key]
        label = item[self.label_key]
        
        # Ensure RGB
        if image.mode != 'RGB':
            image = image.convert('RGB')
        
        image = self.transform(image)
        return image, label

print('HFImageDataset wrapper ready.')

## Benchmark 1: ImageNet-A (Natural Adversarial Examples)

7,500 images that fool ImageNet-trained models. 200 classes (subset of IN-1k).
Requires mapping from IN-A class indices to IN-1k class indices.

In [ ]:
# ============================================================
# ImageNet-A: class index mapping
# ImageNet-A uses 200 of the 1000 ImageNet classes.
# We need to map model outputs (1000-way) to the 200 classes.
# ============================================================

# The 200 ImageNet-A class indices (in IN-1k space)
IMAGENET_A_CLASSES = [
    6, 11, 13, 15, 17, 22, 23, 27, 30, 37, 39, 42, 47, 50, 57, 70, 71, 76, 79, 89,
    90, 94, 96, 97, 99, 105, 107, 108, 110, 113, 124, 125, 130, 132, 143, 144, 150,
    151, 207, 234, 235, 254, 277, 283, 287, 291, 295, 298, 301, 306, 307, 308, 309,
    310, 311, 313, 314, 315, 317, 319, 323, 324, 326, 327, 330, 334, 335, 336, 347,
    361, 363, 372, 378, 386, 397, 400, 401, 402, 404, 407, 411, 416, 417, 420, 425,
    428, 430, 437, 438, 445, 456, 457, 461, 462, 470, 472, 483, 486, 488, 492, 496,
    514, 516, 528, 530, 539, 542, 543, 549, 552, 557, 561, 562, 569, 572, 573, 575,
    579, 589, 606, 607, 609, 614, 626, 627, 640, 641, 642, 643, 658, 668, 677, 682,
    684, 687, 701, 704, 719, 736, 746, 749, 752, 758, 763, 765, 768, 773, 774, 776,
    779, 780, 786, 792, 797, 802, 803, 804, 813, 815, 820, 823, 831, 833, 835, 839,
    845, 847, 850, 859, 862, 870, 879, 880, 888, 890, 897, 900, 907, 913, 924, 932,
    933, 934, 937, 943, 945, 947, 951, 954, 956, 957, 959, 971, 972, 980, 981, 984,
    986, 987, 988
]


@torch.no_grad()
def evaluate_imagenet_a(model, dataloader):
    """Evaluate on ImageNet-A with proper class mapping."""
    correct = 0
    total = 0
    
    for batch_idx, (images, labels) in enumerate(dataloader):
        images = images.to(device)
        labels = labels.to(device)
        
        logits = model(images)  # (B, 1000)
        # Select only the 200 IN-A classes
        logits_a = logits[:, IMAGENET_A_CLASSES]  # (B, 200)
        preds = logits_a.argmax(dim=1)  # (B,) in [0, 199]
        
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
        if (batch_idx + 1) % 20 == 0:
            print(f'    Batch {batch_idx+1}/{len(dataloader)} '
                  f'(acc={100*correct/total:.2f}%)', end='\r')
    
    print()
    return {1: 100.0 * correct / total}

print('ImageNet-A evaluator ready.')

In [ ]:
# ============================================================
# Load ImageNet-A
# ============================================================

print('Loading ImageNet-A from HuggingFace...')
try:
    ds_a = load_dataset('GATE-engine/imagenet-a', split='test')
    dataset_a = HFImageDataset(ds_a, transform_eval, max_samples=5000)
    loader_a = torch.utils.data.DataLoader(dataset_a, batch_size=64, shuffle=False, num_workers=2)
    print(f'ImageNet-A: {len(dataset_a)} samples')
    HAS_IMAGENET_A = True
except Exception as e:
    print(f'Failed to load ImageNet-A: {e}')
    print('Trying alternative source...')
    try:
        ds_a = load_dataset('imagenet_a', split='test', trust_remote_code=True)
        dataset_a = HFImageDataset(ds_a, transform_eval, max_samples=5000)
        loader_a = torch.utils.data.DataLoader(dataset_a, batch_size=64, shuffle=False, num_workers=2)
        print(f'ImageNet-A: {len(dataset_a)} samples')
        HAS_IMAGENET_A = True
    except Exception as e2:
        print(f'Could not load ImageNet-A: {e2}')
        HAS_IMAGENET_A = False

In [ ]:
# ============================================================
# Evaluate ImageNet-A
# ============================================================

results_all = {}  # model_name -> {benchmark -> {metric -> value}}

if HAS_IMAGENET_A:
    print('=== ImageNet-A ===')
    
    print('  Loading ViT-5-Small...')
    model = load_vit5_small()
    r = evaluate_imagenet_a(model, loader_a)
    results_all.setdefault('ViT-5-Small', {})['ImageNet-A'] = r[1]
    print(f'  ViT-5-Small: {r[1]:.2f}%')
    del model; torch.cuda.empty_cache()
    
    print('  Loading DeiT-III-Small...')
    model = load_deit3_small()
    r = evaluate_imagenet_a(model, loader_a)
    results_all.setdefault('DeiT-III-Small', {})['ImageNet-A'] = r[1]
    print(f'  DeiT-III-Small: {r[1]:.2f}%')
    del model; torch.cuda.empty_cache()
else:
    print('Skipping ImageNet-A (dataset not available)')

## Benchmark 2: ImageNet-R (Renditions)

In [ ]:
# ============================================================
# ImageNet-R: 200 classes, same mapping as ImageNet-A
# ============================================================

# ImageNet-R uses the same 200 class subset as ImageNet-A
IMAGENET_R_CLASSES = IMAGENET_A_CLASSES  # Same 200 classes

@torch.no_grad()
def evaluate_imagenet_r(model, dataloader):
    """Evaluate on ImageNet-R with proper class mapping."""
    correct = 0
    total = 0
    
    for batch_idx, (images, labels) in enumerate(dataloader):
        images = images.to(device)
        labels = labels.to(device)
        
        logits = model(images)
        logits_r = logits[:, IMAGENET_R_CLASSES]
        preds = logits_r.argmax(dim=1)
        
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
        if (batch_idx + 1) % 20 == 0:
            print(f'    Batch {batch_idx+1}/{len(dataloader)} '
                  f'(acc={100*correct/total:.2f}%)', end='\r')
    
    print()
    return {1: 100.0 * correct / total}

print('Loading ImageNet-R from HuggingFace...')
try:
    ds_r = load_dataset('GATE-engine/imagenet-r', split='test')
    dataset_r = HFImageDataset(ds_r, transform_eval, max_samples=10000)
    loader_r = torch.utils.data.DataLoader(dataset_r, batch_size=64, shuffle=False, num_workers=2)
    print(f'ImageNet-R: {len(dataset_r)} samples')
    HAS_IMAGENET_R = True
except Exception as e:
    print(f'Failed to load ImageNet-R: {e}')
    try:
        ds_r = load_dataset('imagenet_r', split='test', trust_remote_code=True)
        dataset_r = HFImageDataset(ds_r, transform_eval, max_samples=10000)
        loader_r = torch.utils.data.DataLoader(dataset_r, batch_size=64, shuffle=False, num_workers=2)
        print(f'ImageNet-R: {len(dataset_r)} samples')
        HAS_IMAGENET_R = True
    except Exception as e2:
        print(f'Could not load ImageNet-R: {e2}')
        HAS_IMAGENET_R = False

In [ ]:
if HAS_IMAGENET_R:
    print('=== ImageNet-R ===')
    
    print('  Loading ViT-5-Small...')
    model = load_vit5_small()
    r = evaluate_imagenet_r(model, loader_r)
    results_all.setdefault('ViT-5-Small', {})['ImageNet-R'] = r[1]
    print(f'  ViT-5-Small: {r[1]:.2f}%')
    del model; torch.cuda.empty_cache()
    
    print('  Loading DeiT-III-Small...')
    model = load_deit3_small()
    r = evaluate_imagenet_r(model, loader_r)
    results_all.setdefault('DeiT-III-Small', {})['ImageNet-R'] = r[1]
    print(f'  DeiT-III-Small: {r[1]:.2f}%')
    del model; torch.cuda.empty_cache()
else:
    print('Skipping ImageNet-R (dataset not available)')

## Benchmark 3: ImageNet-Sketch

In [ ]:
# ============================================================
# ImageNet-Sketch: 1000 classes (same as IN-1k)
# ============================================================

print('Loading ImageNet-Sketch from HuggingFace...')
try:
    ds_sk = load_dataset('GATE-engine/imagenet-sketch', split='test')
    dataset_sk = HFImageDataset(ds_sk, transform_eval, max_samples=10000)
    loader_sk = torch.utils.data.DataLoader(dataset_sk, batch_size=64, shuffle=False, num_workers=2)
    print(f'ImageNet-Sketch: {len(dataset_sk)} samples')
    HAS_IMAGENET_SK = True
except Exception as e:
    print(f'Failed to load ImageNet-Sketch: {e}')
    try:
        ds_sk = load_dataset('imagenet_sketch', split='test', trust_remote_code=True)
        dataset_sk = HFImageDataset(ds_sk, transform_eval, max_samples=10000)
        loader_sk = torch.utils.data.DataLoader(dataset_sk, batch_size=64, shuffle=False, num_workers=2)
        print(f'ImageNet-Sketch: {len(dataset_sk)} samples')
        HAS_IMAGENET_SK = True
    except Exception as e2:
        print(f'Could not load ImageNet-Sketch: {e2}')
        HAS_IMAGENET_SK = False

In [ ]:
if HAS_IMAGENET_SK:
    print('=== ImageNet-Sketch ===')
    
    print('  Loading ViT-5-Small...')
    model = load_vit5_small()
    r = evaluate_accuracy(model, loader_sk)
    results_all.setdefault('ViT-5-Small', {})['ImageNet-Sketch'] = r[1]
    print(f'  ViT-5-Small: Top-1={r[1]:.2f}%')
    del model; torch.cuda.empty_cache()
    
    print('  Loading DeiT-III-Small...')
    model = load_deit3_small()
    r = evaluate_accuracy(model, loader_sk)
    results_all.setdefault('DeiT-III-Small', {})['ImageNet-Sketch'] = r[1]
    print(f'  DeiT-III-Small: Top-1={r[1]:.2f}%')
    del model; torch.cuda.empty_cache()
else:
    print('Skipping ImageNet-Sketch (dataset not available)')

## Benchmark 4: ImageNet-C (Corruption Robustness)

15 corruption types x 5 severity levels. We test severity 3 (moderate) and 5 (severe)
using programmatic corruptions applied to CIFAR-100 or a small ImageNet subset.

In [ ]:
# ============================================================
# Programmatic corruptions (no external dataset needed)
# We apply standard ImageNet-C corruptions to clean images
# ============================================================

!pip install -q imagecorruptions

from imagecorruptions import corrupt, get_corruption_names

CORRUPTION_TYPES = get_corruption_names('all')
print(f'Available corruptions ({len(CORRUPTION_TYPES)}): {CORRUPTION_TYPES}')

In [ ]:
# ============================================================
# Create corrupted datasets on-the-fly
# ============================================================

from torchvision import datasets

class CorruptedDataset(torch.utils.data.Dataset):
    """Applies a corruption to images on-the-fly."""
    def __init__(self, base_dataset, corruption_name, severity, transform):
        self.base = base_dataset
        self.corruption_name = corruption_name
        self.severity = severity
        self.transform = transform
    
    def __len__(self):
        return len(self.base)
    
    def __getitem__(self, idx):
        # Get raw image (before normalization)
        if hasattr(self.base, 'data'):  # CIFAR
            img_array = self.base.data[idx]  # numpy HWC uint8
            label = self.base.targets[idx]
        else:
            img, label = self.base[idx]
            img_array = np.array(img)
        
        # Apply corruption (expects numpy HWC uint8)
        if img_array.shape[0] < img_array.shape[2]:  # CHW -> HWC
            img_array = img_array.transpose(1, 2, 0)
        
        corrupted = corrupt(img_array, corruption_name=self.corruption_name, severity=self.severity)
        
        # Convert back to PIL for transforms
        img_pil = Image.fromarray(corrupted)
        img_tensor = self.transform(img_pil)
        
        return img_tensor, label


# Base dataset: CIFAR-100 (clean)
# Note: CIFAR-100 is 32x32 — we resize to 224x224 in transform.
# Corruption patterns are architecture-dependent.
cifar100_raw = datasets.CIFAR100(root='./data', train=False, download=True)

# Use 500 samples per corruption for speed
NUM_CORRUPT_SAMPLES = 500

print(f'Will test {len(CORRUPTION_TYPES)} corruptions x 2 severities (3, 5) = {len(CORRUPTION_TYPES)*2} settings')
print(f'{NUM_CORRUPT_SAMPLES} samples per setting')

In [ ]:
# ============================================================
# Run corruption robustness evaluation
# ============================================================

def evaluate_corruptions(load_fn, model_name, severities=[3, 5]):
    """Evaluate a model on all corruptions at given severities."""
    print(f'\nEvaluating {model_name} on corruptions...')
    model = load_fn()
    
    results = {}  # corruption_name -> {severity -> accuracy}
    
    for corruption in CORRUPTION_TYPES:
        results[corruption] = {}
        for severity in severities:
            try:
                ds = CorruptedDataset(cifar100_raw, corruption, severity, transform_eval)
                subset = torch.utils.data.Subset(ds, list(range(min(NUM_CORRUPT_SAMPLES, len(ds)))))
                loader = torch.utils.data.DataLoader(subset, batch_size=64, shuffle=False, num_workers=0)
                
                correct = 0
                total = 0
                with torch.no_grad():
                    for images, labels in loader:
                        images = images.to(device)
                        labels = labels.to(device)
                        preds = model(images).argmax(dim=1)
                        correct += (preds == labels).sum().item()
                        total += labels.size(0)
                
                acc = 100.0 * correct / total
                results[corruption][severity] = acc
            except Exception as e:
                print(f'    Skipping {corruption} sev={severity}: {e}')
                results[corruption][severity] = None
        
        s3 = results[corruption].get(3, None)
        s5 = results[corruption].get(5, None)
        s3_str = f'{s3:.1f}%' if s3 is not None else 'N/A'
        s5_str = f'{s5:.1f}%' if s5 is not None else 'N/A'
        print(f'    {corruption:>25s}: sev3={s3_str:>6s}  sev5={s5_str:>6s}')
    
    del model
    torch.cuda.empty_cache()
    return results

print('Running corruption evaluation...')
corrupt_vit5 = evaluate_corruptions(load_vit5_small, 'ViT-5-Small')
corrupt_deit = evaluate_corruptions(load_deit3_small, 'DeiT-III-Small')

In [ ]:
# ============================================================
# Corruption results table
# ============================================================

print('\n' + '=' * 90)
print('CORRUPTION ROBUSTNESS COMPARISON')
print('=' * 90)
print(f"{'Corruption':>25s} | {'ViT5 sev3':>10s} {'D3 sev3':>10s} {'diff':>7s} | "
      f"{'ViT5 sev5':>10s} {'D3 sev5':>10s} {'diff':>7s}")
print('-' * 90)

diffs_s3 = []
diffs_s5 = []

for corruption in CORRUPTION_TYPES:
    v5_s3 = corrupt_vit5[corruption].get(3)
    d3_s3 = corrupt_deit[corruption].get(3)
    v5_s5 = corrupt_vit5[corruption].get(5)
    d3_s5 = corrupt_deit[corruption].get(5)
    
    def fmt(v):
        return f'{v:.1f}%' if v is not None else 'N/A'
    def diff(a, b):
        if a is not None and b is not None:
            return f'{a-b:+.1f}'
        return 'N/A'
    
    if v5_s3 is not None and d3_s3 is not None:
        diffs_s3.append(v5_s3 - d3_s3)
    if v5_s5 is not None and d3_s5 is not None:
        diffs_s5.append(v5_s5 - d3_s5)
    
    print(f'{corruption:>25s} | {fmt(v5_s3):>10s} {fmt(d3_s3):>10s} {diff(v5_s3, d3_s3):>7s} | '
          f'{fmt(v5_s5):>10s} {fmt(d3_s5):>10s} {diff(v5_s5, d3_s5):>7s}')

print('-' * 90)
if diffs_s3:
    print(f'Average delta sev3: {np.mean(diffs_s3):+.2f}%')
if diffs_s5:
    print(f'Average delta sev5: {np.mean(diffs_s5):+.2f}%')
print(f'ViT-5 wins (sev3): {sum(1 for d in diffs_s3 if d > 0)}/{len(diffs_s3)}')
print(f'ViT-5 wins (sev5): {sum(1 for d in diffs_s5 if d > 0)}/{len(diffs_s5)}')

In [ ]:
# ============================================================
# Corruption robustness bar chart
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('Corruption Robustness: ViT-5-Small vs DeiT-III-Small', fontsize=14, fontweight='bold')

for ax, sev in zip(axes, [3, 5]):
    corruptions = [c for c in CORRUPTION_TYPES 
                   if corrupt_vit5[c].get(sev) is not None and corrupt_deit[c].get(sev) is not None]
    v5_accs = [corrupt_vit5[c][sev] for c in corruptions]
    d3_accs = [corrupt_deit[c][sev] for c in corruptions]
    
    x = np.arange(len(corruptions))
    w = 0.35
    
    ax.bar(x - w/2, v5_accs, w, color='#e74c3c', alpha=0.8, label='ViT-5-Small')
    ax.bar(x + w/2, d3_accs, w, color='#3498db', alpha=0.8, label='DeiT-III-Small')
    
    ax.set_xlabel('Corruption Type')
    ax.set_ylabel('Top-1 Accuracy (%)')
    ax.set_title(f'Severity {sev}')
    ax.set_xticks(x)
    ax.set_xticklabels(corruptions, rotation=45, ha='right', fontsize=8)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('exp5_corruption_robustness.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: exp5_corruption_robustness.png')

In [ ]:
# ============================================================
# Final summary
# ============================================================

print('=' * 70)
print('ROBUSTNESS EVALUATION SUMMARY')
print('=' * 70)
print()

print(f"{'Benchmark':>20s} | {'ViT-5-Small':>12s} {'DeiT-III-Small':>15s} {'Delta':>8s}")
print('-' * 60)

for benchmark in ['ImageNet-A', 'ImageNet-R', 'ImageNet-Sketch']:
    v5 = results_all.get('ViT-5-Small', {}).get(benchmark)
    d3 = results_all.get('DeiT-III-Small', {}).get(benchmark)
    if v5 is not None and d3 is not None:
        print(f'{benchmark:>20s} | {v5:>11.2f}% {d3:>14.2f}% {v5-d3:>+7.2f}%')
    else:
        print(f'{benchmark:>20s} | {"N/A":>12s} {"N/A":>15s}')

if diffs_s3:
    v5_avg_s3 = np.mean([corrupt_vit5[c][3] for c in CORRUPTION_TYPES if corrupt_vit5[c].get(3) is not None])
    d3_avg_s3 = np.mean([corrupt_deit[c][3] for c in CORRUPTION_TYPES if corrupt_deit[c].get(3) is not None])
    print(f'{"Avg Corruption (s3)":>20s} | {v5_avg_s3:>11.2f}% {d3_avg_s3:>14.2f}% {v5_avg_s3-d3_avg_s3:>+7.2f}%')

if diffs_s5:
    v5_avg_s5 = np.mean([corrupt_vit5[c][5] for c in CORRUPTION_TYPES if corrupt_vit5[c].get(5) is not None])
    d3_avg_s5 = np.mean([corrupt_deit[c][5] for c in CORRUPTION_TYPES if corrupt_deit[c].get(5) is not None])
    print(f'{"Avg Corruption (s5)":>20s} | {v5_avg_s5:>11.2f}% {d3_avg_s5:>14.2f}% {v5_avg_s5-d3_avg_s5:>+7.2f}%')

print()
print('NOTE: The ViT-5 paper reports NONE of these benchmarks.')
print('For a backbone paper in 2026, this is a significant omission.')
print('If ViT-5 gains clean accuracy but loses robustness,')
print('the "drop-in upgrade" claim is misleading.')